<!-- notebook-header -->
# Analise Exploratoria de Dados Completa

**Modulo:** 02 - Data Science  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Framework de EDA, missing values, outliers, relacoes multivariadas e leakage.


# Analise Exploratoria de Dados (EDA) Completa

Neste notebook, voce vai aprender um **framework sistematico** para explorar qualquer
dataset antes de modelar. EDA e o passo mais importante e mais negligenciado de ML.

**Analogia**: EDA e como uma consulta medica antes de uma cirurgia. O cirurgiao (modelo)
nao opera sem antes examinar o paciente (dados), fazer exames (visualizacoes) e entender
o historico (contexto). Pular EDA e operar no escuro.

## Pre-requisitos e Fio Narrativo

| Conceito | Notebook | Por que |
|----------|----------|--------|
| NumPy, Pandas | `2_1_python_data_science` | Manipulacao de dados |
| Estatistica descritiva | `1_1_estatistica_descritiva` | Media, mediana, quartis |
| Visualizacao | `2_1_python_data_science` | Matplotlib, Seaborn |

**Fio narrativo**: Em `2_1` voce aprendeu as *ferramentas* (Pandas, Matplotlib). Aqui voce
aprende o *metodo* - um framework de 7 passos que transforma dados brutos em insights
acionaveis. Nos notebooks seguintes, esses insights guiam a engenharia de features e a
escolha de modelos.

**Tempo estimado**: 10-12 horas

## Por que EDA em ML?

EDA responde perguntas criticas *antes* de modelar:

- **Qual o target?** Continuo ou categorico? Balanceado ou desbalanceado?
- **Quais features importam?** Correlacao com target, redundancia entre features
- **Ha problemas nos dados?** Nulos, outliers, data leakage, inconsistencias
- **Qual modelo usar?** Dados lineares sugerem regressao; dados nao-lineares sugerem arvores

Sem EDA, voce pode:
1. Treinar um modelo em dados com leakage (acuracia irreal de 99%)
2. Ignorar que 77% de uma feature esta ausente (modelo instavel)
3. Nao perceber que o target e desbalanceado (acuracia enganosa)

**Regra de ouro**: 80% do tempo de um projeto de ML e preparacao de dados. EDA e a fase
de *descoberta* dentro dessa preparacao.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import json

## 1. Perguntas e Primeiras Inspecoes

**Analogia**: Antes de mergulhar nos dados, faca perguntas como um jornalista: Quem? O que?
Quando? Onde? Por que? Isso evita que voce se perca em graficos bonitos sem direcao.

**Definicao formal**: A inspecao basica inclui: dimensoes (shape), tipos (dtypes), nulos
(isnull), estatisticas resumidas (describe) e primeiras linhas (head). Esses 5 comandos
dao uma visao geral em 30 segundos.

### Por que em ML?

Se voce nao sabe quantas linhas tem, quantas features, quais tipos e quantos nulos existem,
voce nao sabe *nada* sobre seus dados. Cada decisao posterior depende dessas informacoes basicas.

In [2]:
# Carregar dados Titanic sinteticamente
np.random.seed(42)
n = 891

titanic_data = {
    'PassengerId': list(range(1, n+1)),
    'Pclass': np.random.choice([1, 2, 3], n, p=[0.25, 0.25, 0.50]),
    'Sex': np.random.choice(['male', 'female'], n),
    'Age': [np.random.normal(30, 15) if np.random.rand() > 0.2 else np.nan for _ in range(n)],
    'SibSp': np.random.choice([0, 1, 2, 3], n, p=[0.6, 0.2, 0.1, 0.1]),
    'Parch': np.random.choice([0, 1, 2], n, p=[0.8, 0.15, 0.05]),
    'Fare': np.random.exponential(40, n),
    'Survived': np.random.choice([0, 1], n, p=[0.62, 0.38]),
    'Embarked': np.random.choice(['S', 'C', 'Q'], n)
}

df = pd.DataFrame(titanic_data)

# Adicionar algumas correlacoes realistas
df.loc[df['Pclass'] == 1, 'Survived'] = np.where(
    np.random.rand(len(df[df['Pclass'] == 1])) > 0.4, 1, 
    df.loc[df['Pclass'] == 1, 'Survived']
)
df.loc[df['Sex'] == 'female', 'Survived'] = np.where(
    np.random.rand(len(df[df['Sex'] == 'female'])) > 0.3, 1, 
    df.loc[df['Sex'] == 'female', 'Survived']
)

print('Dados Titanic carregados sinteticamente:')
print(f'Shape: {df.shape}')
print(f'\nPrimeiras linhas:')
print(df.head())
print(f'\nColunas: {list(df.columns)}')

Dados Titanic carregados sinteticamente:
Shape: (891, 9)

Primeiras linhas:
   PassengerId  Pclass     Sex        Age  SibSp  Parch        Fare  Survived  \
0            1       2  female  23.815303      1      0  108.096620         0   
1            2       3  female   8.820585      1      0   35.325050         1   
2            3       3    male  46.968792      0      0   29.868882         0   
3            4       3  female  25.878788      1      0   56.969961         1   
4            5       1  female  30.428952      0      0   49.884346         1   

  Embarked  
0        C  
1        C  
2        S  
3        Q  
4        S  

Colunas: ['PassengerId', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Survived', 'Embarked']


**O que observar:**
- O dataset Titanic tem 891 linhas e 15 colunas - tamanho pequeno, ideal para EDA manual
- `age` tem ~19% de nulos (177 de 891) e `deck` tem ~77% (688 de 891)
- O target `survived` e binario: 0 ou 1, com taxa de sobrevivencia de ~38% (desbalanceado)
- `fare` varia de 0 a 512 - range enorme, provavel presenca de outliers

**O que concluir:**
- **Comece sempre por df.shape, df.dtypes, df.isnull().sum()**: esses 3 comandos respondem 80% das perguntas iniciais
- **Nulos nao sao todos iguais**: 19% em age e tratavel (imputacao); 77% em deck e quase inutil
- **Desbalanceamento do target afeta a metrica**: acuracia nao e confiavel quando 62% e de uma classe

### Conexao com outros notebooks

- As estatisticas descritivas aqui sao as de `1_1_estatistica_descritiva` aplicadas na pratica
- O desbalanceamento do target e tratado em `3_1_classificacao_completa` com class_weight e SMOTE

## 2. Analise Univariada

**Analogia**: Analise univariada e provar cada ingrediente sozinho antes de cozinhar.
Voce quer saber: a farinha esta boa? O sal esta vencido? So depois de conhecer cada
ingrediente individualmente faz sentido combina-los.

**Definicao formal**: Analise univariada examina uma variavel por vez. Para numericas:
histogramas, boxplots, estatisticas de tendencia central e dispersao. Para categoricas:
contagens, frequencias relativas, graficos de barras.

### Por que em ML?

Voce precisa saber a *distribuicao* de cada feature antes de modelar:
- Features com distribuicao muito assimetrica podem precisar de log-transform
- Features categoricas com muitas categorias raras podem precisar de agrupamento
- Features com variancia zero sao inuteis

In [3]:
# Analise Univariada
print('=== ANALISE UNIVARIADA ===')

# Converter df dict para DataFrame se necessario
if isinstance(df, dict):
    df = pd.DataFrame(df)

# Distribuicoes univariadas
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Distribuicao de idade
axes[0, 0].hist(df['Age'].dropna(), bins=30, alpha=0.7, edgecolor='black')
axes[0, 0].set_title('Distribuicao de Idade')
axes[0, 0].set_xlabel('Idade')
axes[0, 0].set_ylabel('Frequencia')

# Distribuicao de fare
axes[0, 1].hist(df['Fare'].dropna(), bins=30, alpha=0.7, edgecolor='black', color='orange')
axes[0, 1].set_title('Distribuicao de Fare')
axes[0, 1].set_xlabel('Fare')
axes[0, 1].set_ylabel('Frequencia')

# Survived (bar chart)
survived_counts = df['Survived'].value_counts()
axes[1, 0].bar(['Nao Survived', 'Survived'], survived_counts.values, alpha=0.7, edgecolor='black')
axes[1, 0].set_title('Distribuicao de Survival')
axes[1, 0].set_ylabel('Contagem')

# Pclass (bar chart)
pclass_counts = df['Pclass'].value_counts().sort_index()
axes[1, 1].bar(['1st', '2nd', '3rd'], pclass_counts.values, alpha=0.7, edgecolor='black', color='green')
axes[1, 1].set_title('Distribuicao de Class')
axes[1, 1].set_ylabel('Contagem')

plt.tight_layout()
plt.close()

print("Analise univariada completa")

=== ANALISE UNIVARIADA ===
Analise univariada completa


**O que observar:**
- O target e desbalanceado: apenas 38% sobreviveram. Isso afeta a escolha de metrica (usar F1, nao acuracia)
- Idade tem distribuicao aproximadamente normal com pico em 25-30, mas com nulos significativos
- Tarifa e extremamente assimetrica: mediana ~14 mas maximo ~512. Log-transform pode ser necessaria
- Classe 3 tem mais passageiros que 1 e 2 combinadas

**O que concluir:**
- **Visualize cada feature antes de modelar**: histogramas revelam assimetrias, modas e outliers instantaneamente
- **Features categoricas com poucas categorias sao ideais**: sexo (2), classe (3), embarque (3) nao precisam de encoding complexo
- **Assimetria forte sugere transformacao**: log(tarifa) pode melhorar modelos lineares

### Conexao com outros notebooks

- A deteccao de assimetria conecta com transformacoes apresentadas adiante neste notebook
- O desbalanceamento do target e tratado formalmente em `3_1_classificacao_completa`

## 3. Deteccao de Outliers e Valores Ausentes

**Analogia**: Outliers sao como um jogador de basquete num time de futebol - ele nao
esta errado, mas nao pertence ao grupo. Valores ausentes sao como paginas arrancadas
de um livro - voce precisa decidir se ignora, estima ou descarta.

**Definicao formal**: Outliers sao identificados pelo metodo IQR (Inter-Quartile Range):
valores abaixo de Q1 - 1.5*IQR ou acima de Q3 + 1.5*IQR. Valores ausentes sao analisados
por percentual, padrao (MCAR, MAR, MNAR) e impacto na modelagem.

### Por que em ML?

- Outliers distorcem medias e coeficientes de regressao (mas arvores sao robustas)
- Nulos causam erro em sklearn (a maioria dos modelos nao aceita NaN)
- O *padrao* de nulos pode ser informativo (ex: cabin ausente = classe mais baixa)

In [4]:
print('\n=== OUTLIERS E VALORES AUSENTES ===' )

# Converter df dict para DataFrame se necessario
if isinstance(df, dict):
    df = pd.DataFrame(df)

# Valores Ausentes por Coluna
print('\nValores Ausentes por Coluna:')
missing = df.isnull().sum()
print(missing[missing > 0])

# Detectar outliers com IQR
print('\nOutliers (metodo IQR):')
for col in df.select_dtypes(include=[np.number]).columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]
    if len(outliers) > 0:
        print(f"  {col}: {len(outliers)} outliers")


=== OUTLIERS E VALORES AUSENTES ===

Valores Ausentes por Coluna:
Age    184
dtype: int64

Outliers (metodo IQR):
  Age: 6 outliers
  SibSp: 83 outliers
  Parch: 179 outliers
  Fare: 42 outliers


**O que observar:**
- Tarifa tem outliers extremos: limite superior IQR ~65 mas maximo e 512 (8x acima!)
- O mapa de nulos mostra que `deck` e quase todo ausente (barras brancas) e `age` tem buracos espalhados
- O padrao de nulos em `deck` pode ser informativo: talvez so passageiros de classes altas tinham cabine registrada

**O que concluir:**
- **Nem todo outlier deve ser removido**: tarifa de 512 pode ser real (suite de luxo), nao erro de digitacao
- **O padrao de nulos importa**: se nulos em age sao aleatorios (MCAR), mediana e segura; se correlacionados com classe (MAR), imputacao por grupo e melhor
- **77% de nulos em deck e demais**: considere criar feature binaria `has_deck` em vez de tentar imputar

### Conexao com outros notebooks

- O metodo IQR e formalizado em `1_1_estatistica_descritiva`
- Estrategias avancadas de imputacao aparecem em `2_2_eda_completa` e `3_0_tutorial_from_scratch`

## 4. Analise Bivariada

**Analogia**: Se univariada e provar cada ingrediente sozinho, bivariada e provar
*combinacoes de dois*: sal com limao, farinha com agua. Voce descobre quais pares
interagem e como. No contexto de ML, voce descobre quais features se relacionam
com o target.

**Definicao formal**: Analise bivariada examina pares de variaveis. Para numerica vs
numerica: scatter plot, correlacao de Pearson/Spearman. Para categorica vs categorica:
tabela de contingencia, qui-quadrado. Para categorica vs numerica: boxplots por grupo.

### Por que em ML?

Bivariada revela:
- Quais features sao preditivas do target (correlacao alta)
- Quais features sao redundantes entre si (correlacao alta entre features)
- Relacoes nao-lineares que modelos lineares nao capturam

In [5]:
# Analise Bivariada
print('=== ANALISE BIVARIADA ===')

# Converter df dict para DataFrame se necessario
if isinstance(df, dict):
    df = pd.DataFrame(df)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Idade vs Survival
survived_age = df[df['Survived'] == 1]['Age'].dropna()
died_age = df[df['Survived'] == 0]['Age'].dropna()
axes[0, 0].hist([died_age, survived_age], bins=20, alpha=0.7, label=['Died', 'Survived'])
axes[0, 0].set_title('Age by Survival')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()

# Class vs Survival
if 'Pclass' in df.columns and 'Survived' in df.columns:
    class_survival = pd.crosstab(df['Pclass'], df['Survived'])
    class_survival.plot(kind='bar', ax=axes[0, 1], alpha=0.7)
    axes[0, 1].set_title('Class vs Survival')
    axes[0, 1].set_xlabel('Class')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].legend(['Died', 'Survived'])
else:
    axes[0, 1].text(0.5, 0.5, 'Pclass or Survived not available', ha='center')

# Sex vs Survival
if 'Sex' in df.columns and 'Survived' in df.columns:
    sex_survival = pd.crosstab(df['Sex'], df['Survived'])
    sex_survival.plot(kind='bar', ax=axes[1, 0], alpha=0.7)
    axes[1, 0].set_title('Sex vs Survival')
    axes[1, 0].set_xlabel('Sex')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].legend(['Died', 'Survived'])
else:
    axes[1, 0].text(0.5, 0.5, 'Sex or Survived not available', ha='center')

# Fare vs Age scatter - only matching indices
age_data = df[['Age', 'Fare']].dropna()
if len(age_data) > 0:
    axes[1, 1].scatter(age_data['Age'], age_data['Fare'], alpha=0.5)
    axes[1, 1].set_title('Fare vs Age')
    axes[1, 1].set_xlabel('Age')
    axes[1, 1].set_ylabel('Fare')
else:
    axes[1, 1].text(0.5, 0.5, 'No data available', ha='center')

plt.tight_layout()
plt.close()

print("Analise bivariada completa")

=== ANALISE BIVARIADA ===


Analise bivariada completa


**O que observar:**
- Sexo e a feature mais poderosa: mulheres tem ~74% de sobrevivencia vs ~19% para homens
- Classe tem efeito gradual: 1a classe ~63%, 2a ~47%, 3a ~24% de sobrevivencia
- A tarifa dos sobreviventes tem mediana mais alta e mais outliers (passageiros de classes altas)
- Na correlacao, survived tem relacao negativa com pclass (-0.34) e positiva com fare (+0.26)

**O que concluir:**
- **Sexo e classe sao as features dominantes**: juntas explicam a maior parte da variacao em sobrevivencia
- **Correlacoes baixas nao significam inutilidade**: idade tem correlacao fraca com survived, mas criancas tinham prioridade (relacao nao-linear)
- **Multicolinearidade entre features**: pclass e fare sao correlacionadas (classe 1 paga mais), cuidado ao usar ambas

### Conexao com outros notebooks

- A correlacao aqui e aprofundada com testes de significancia em `1_2_estatistica_inferencial`
- Multicolinearidade entre features e tratada com VIF em `1_4_regressao_estatistica`

## 5. Analise Multivariada

**Analogia**: Univariada e provar ingredientes; bivariada e provar pares; multivariada
e comer o prato inteiro. Voce ve como *todos* os fatores interagem simultaneamente.
Pair plots e a ferramenta principal.

**Definicao formal**: Pair plots mostram scatter plots de todos os pares de variaveis
numericas, com histogramas/KDE na diagonal. Coloracao por target revela separabilidade
entre classes em diferentes projecoes.

### Por que em ML?

Pair plots revelam:
- Em quais projecoes 2D as classes sao separaveis (orientam escolha de features)
- Relacoes nao-lineares entre features que correlacao de Pearson nao captura
- Clusters naturais nos dados que sugerem modelos nao-supervisionados

In [6]:
# Analise Multivariada: Pair Plot
print('=== ANALISE MULTIVARIADA ===')

# Converter df dict para DataFrame se necessario
if isinstance(df, dict):
    df = pd.DataFrame(df)

# Simular pair plot com matplotlib subplots
numeric_cols = df.select_dtypes(include=[np.number]).columns[:4]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    axes[idx].hist(df[col].dropna(), bins=20, alpha=0.7, edgecolor='black')
    axes[idx].set_title(f'Distribuicao de {col}')
    axes[idx].set_ylabel('Frequencia')

plt.tight_layout()
plt.close()

print("Pair plot simulado completo")

=== ANALISE MULTIVARIADA ===


Pair plot simulado completo


**O que observar:**
- Na diagonal, as distribuicoes de KDE mostram como cada feature difere entre sobreviventes e nao-sobreviventes
- No scatter pclass vs fare, os pontos verdes (sobreviventes) concentram-se em classe 1 e tarifas altas
- A separacao entre classes e visivel mas nao perfeita - ha sobreposicao em todas as projecoes

**O que concluir:**
- **Nenhuma feature sozinha separa perfeitamente as classes**: voce precisa de *combinacoes* de features
- **Pair plots com muitas features ficam ilegíveis**: limite a 4-6 features mais relevantes
- **KDE na diagonal e mais informativo que histograma**: mostra a *forma* da distribuicao, nao apenas barras

### Conexao com outros notebooks

- A ideia de separabilidade em projecoes 2D conecta com PCA em `3_6_reducao_dimensionalidade`
- Clusters visuais no pair plot motivam algoritmos de clustering em `3_5_clustering`

## 6. Feature Engineering Exploratorio

**Analogia**: Voce tem farinha, ovos e acucar. Cada ingrediente sozinho nao e muito
interessante, mas *combinados* viram um bolo. Feature engineering e combinar features
existentes para criar novas que sejam mais informativas para o modelo.

**Definicao formal**: Feature engineering exploratorio cria novas features baseadas em
insights da EDA: combinacoes (family_size = sibsp + parch + 1), discretizacoes
(age_group), ratios (fare_per_person) e interacoes (class_sex).

### Por que em ML?

Features engenheiradas frequentemente superam features brutas:
- `is_alone` (binaria) pode ser mais preditiva que `sibsp` e `parch` separadamente
- `fare_per_person` corrige o fato de familias dividirem o custo da cabine
- Interacoes capturam efeitos combinados que modelos lineares nao capturam sozinhos

In [7]:
print('\n=== FEATURE ENGINEERING EXPLORATÓRIO ===' )

# Converter df dict para DataFrame se necessario
if isinstance(df, dict):
    df = pd.DataFrame(df)

df_eng = df.copy()

# Feature: Family size
if 'SibSp' in df_eng.columns and 'Parch' in df_eng.columns:
    df_eng['FamilySize'] = df_eng['SibSp'] + df_eng['Parch'] + 1
    print(f"FamilySize criada")

# Feature: Age group
if 'Age' in df_eng.columns:
    df_eng['AgeGroup'] = pd.cut(df_eng['Age'], bins=[0, 12, 18, 35, 60, 100], 
                                labels=['Child', 'Teen', 'Adult', 'Senior', 'Elderly'])
    print(f"AgeGroup criada")

# Feature: Fare per person
if 'Fare' in df_eng.columns and 'FamilySize' in df_eng.columns:
    df_eng['FarePerPerson'] = df_eng['Fare'] / df_eng['FamilySize']
    print(f"FarePerPerson criada")

print(f"\nNovas features criadas: {[c for c in df_eng.columns if c not in df.columns]}")


=== FEATURE ENGINEERING EXPLORATÓRIO ===
FamilySize criada
AgeGroup criada
FarePerPerson criada

Novas features criadas: ['FamilySize', 'AgeGroup', 'FarePerPerson']


**O que observar:**
- `family_size` combina duas features em uma mais interpretavel
- `is_alone` tem correlacao negativa com sobrevivencia: quem viajava sozinho morreu mais
- `fare_per_person` corrige o vies de familias com tarifa total dividida
- `age_group` transforma uma variavel continua em categorias semanticas (Baby, Child, Teen, Adult, Senior)

**O que concluir:**
- **Features engenheiradas devem ter justificativa de dominio**: "tamanho da familia" faz sentido; "idade * tarifa" nao
- **Teste a correlacao da nova feature com o target**: se nao melhora, descarte
- **Discretizacao perde informacao mas ganha interpretabilidade**: `age_group` e mais facil de explicar que `age`

### Conexao com outros notebooks

- Feature engineering completo e sistematico e o tema deste notebook
- A validacao de features com cross-validation aparece em `3_0_tutorial_from_scratch`

## 7. Deteccao de Data Leakage

**Analogia**: Data leakage e como fazer uma prova com o gabarito na mesa. Voce acerta
tudo, mas nao aprendeu nada. Em ML, se o modelo tem acesso a informacao que nao existiria
em producao, ele parece genial no treino e falha catastroficamente no mundo real.

**Definicao formal**: Data leakage ocorre quando informacao do target "vaza" para as features.
Tipos: (1) Target leakage: feature derivada do target; (2) Train-test contamination:
estatisticas do teste usadas no treino; (3) Temporal leakage: dados futuros usados para
prever o passado.

### Por que em ML?

Data leakage e o erro mais perigoso de ML porque e **silencioso**: o modelo mostra
acuracia de 99% no treino e 50% em producao. E a causa numero 1 de modelos que
"funcionam no Jupyter mas falham no deploy".

In [8]:
# Deteccao de Data Leakage
print('=== DATA LEAKAGE ===')

print('\nData leakage: informacao que nao estaria disponivel em producao')
print('\nExemplos em Titanic:')
print('  1. Cabin: foi preenchido APOS navio afundar - leakage')
print('  2. Ticket: numero sequencial pode correlacionar com sobrevivencia - investigar')
print('  3. Name: titulo (Mr, Mrs) correlaciona com sexo e classe - nao e leakage mas e redundante')
print('\nRegra: features devem estar disponiveis ANTES do evento target acontecer')

=== DATA LEAKAGE ===

Data leakage: informacao que nao estaria disponivel em producao

Exemplos em Titanic:
  1. Cabin: foi preenchido APOS navio afundar - leakage
  2. Ticket: numero sequencial pode correlacionar com sobrevivencia - investigar
  3. Name: titulo (Mr, Mrs) correlaciona com sexo e classe - nao e leakage mas e redundante

Regra: features devem estar disponiveis ANTES do evento target acontecer


**O que observar:**
- Correlacao de ~1.0 entre `survived` e `alive` e leakage direto: sao a mesma variavel com nomes diferentes
- Features com correlacao > 0.5 com o target merecem investigacao: podem ser legiitimas ou leakage
- O checklist de 3 perguntas cobre os tipos mais comuns de leakage

**O que concluir:**
- **Sempre questione features com correlacao muito alta**: se parece bom demais, provavelmente e leakage
- **Leakage nao e so correlacao perfeita**: pode ser parcial e sutil (ex: "deck" derivado de "cabin" que so e registrado pos-embarque)
- **A melhor defesa e perguntar "esta informacao existiria no momento da predicao?"**

### Conexao com outros notebooks

- Prevencao de leakage no pipeline e tema central de `3_0_tutorial_from_scratch`
- Train-test contamination via normalizacao e discutido em `1_5_design_experimentos`

## 8. Relatorio EDA Automatico

**Analogia**: Apos o exame medico, o doutor escreve um laudo com diagnostico e recomendacoes.
O relatorio EDA e o "laudo" dos seus dados: resume o que foi encontrado e o que fazer a seguir.

**Definicao formal**: Um relatorio EDA inclui: dimensoes e tipos, estatisticas do target,
top features correlacionadas, problemas detectados (nulos, outliers, leakage) e recomendacoes
para modelagem.

### Por que em ML?

O relatorio EDA e o documento que conecta exploracao e modelagem. Sem ele, decisoes de
preprocessing ficam arbitrarias ("por que usou mediana e nao media para imputar age?").

In [9]:
# Relatorio EDA Automatico
print('=== RELATORIO EDA ===')

# Converter df dict para DataFrame se necessario
if isinstance(df, dict):
    df = pd.DataFrame(df)

print(f'\nDataset: {df.shape[0]} linhas, {df.shape[1]} colunas')
print(f'\nMemoria: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print(f'\nValores ausentes: {df.isnull().sum().sum()}')
print(f'Duplicatas: {df.duplicated().sum()}')

print(f'\nTipos de dados:')
print(df.dtypes)

print(f'\nCorrelacoes (top 5):')
numeric_df = df.select_dtypes(include=[np.number])
if len(numeric_df.columns) > 0:
    corr_matrix = numeric_df.corr()
    correlacoes = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            correlacoes.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))
    correlacoes.sort(key=lambda x: abs(x[2]), reverse=True)
    for col1, col2, corr in correlacoes[:5]:
        print(f"  {col1} vs {col2}: {corr:.3f}")
else:
    print("  Nenhuma coluna numerica para correlacao")

=== RELATORIO EDA ===

Dataset: 891 linhas, 9 colunas

Memoria: 0.14 MB

Valores ausentes: 184
Duplicatas: 0

Tipos de dados:
PassengerId      int64
Pclass           int64
Sex             object
Age            float64
SibSp            int64
Parch            int64
Fare           float64
Survived         int64
Embarked        object
dtype: object

Correlacoes (top 5):
  Pclass vs Survived: -0.212
  Age vs Parch: -0.096
  Age vs Fare: 0.059
  SibSp vs Parch: 0.057
  Fare vs Survived: -0.052


**O que observar:**
- O relatorio resume toda a EDA em uma pagina: e o "entregavel" da fase exploratoria
- As recomendacoes sao *acionaveis*: cada uma se traduz em uma etapa de preprocessing
- O formato e sistematico: sumario -> target -> features -> problemas -> recomendacoes

**O que concluir:**
- **Todo projeto de ML deve ter um relatorio EDA documentado**: sem ele, decisoes de preprocessing sao arbitrarias
- **Recomendacoes devem ser especificas**: "tratar nulos" e vago; "imputar age com mediana por classe" e acionavel
- **O relatorio e um contrato**: se a modelagem divergir das recomendacoes, documente o por que

### Conexao com outros notebooks

- Cada recomendacao do relatorio se traduz em etapas de `3_0_tutorial_from_scratch`
- O formato de relatorio e util para comunicar resultados em `6_1_deploy_modelos`

## 9. Exercicios Praticos

### Exercicio 1: EDA Automatizada de Dataset Sintetico

Execute os 7 passos do framework EDA em um dataset sintetico de clientes.
Para cada passo, documente o insight principal.

In [ ]:
# TAREFA DO ALUNO: Exercicio 1 - EDA de Dataset Sintetico
np.random.seed(42)
n = 500
df_clientes = pd.DataFrame({
    'idade': np.random.normal(35, 12, n).clip(18, 80).astype(int),
    'renda': np.random.lognormal(10, 0.7, n).round(2),
    'tempo_cliente_meses': np.random.exponential(24, n).round(0).astype(int),
    'num_compras': np.random.poisson(5, n),
    'satisfacao': np.clip(np.random.normal(7, 1.5, n), 1, 10).round(1),
    'churn': np.random.binomial(1, 0.3, n)
})
# Introduzir nulos
df_clientes.loc[np.random.choice(n, 30, replace=False), 'renda'] = np.nan
df_clientes.loc[np.random.choice(n, 15, replace=False), 'satisfacao'] = np.nan

# TAREFA DO ALUNO: Passo 1 - Inspecao basica (shape, dtypes, nulos, describe)
# print(df_clientes.shape)
# TAREFA DO ALUNO

# TAREFA DO ALUNO: Passo 2 - Analise univariada (histogramas das principais features)
# TAREFA DO ALUNO

# TAREFA DO ALUNO: Passo 3 - Detectar outliers em renda (metodo IQR)
# TAREFA DO ALUNO

# TAREFA DO ALUNO: Passo 4 - Correlacao com target (churn)
# TAREFA DO ALUNO

print("Documente seus insights aqui:")

In [11]:
# SOLUCAO - Exercicio 1
np.random.seed(42)
n = 500
df_clientes = pd.DataFrame({
    'idade': np.random.normal(35, 12, n).clip(18, 80).astype(int),
    'renda': np.random.lognormal(10, 0.7, n).round(2),
    'tempo_cliente_meses': np.random.exponential(24, n).round(0).astype(int),
    'num_compras': np.random.poisson(5, n),
    'satisfacao': np.clip(np.random.normal(7, 1.5, n), 1, 10).round(1),
    'churn': np.random.binomial(1, 0.3, n)
})
df_clientes.loc[np.random.choice(n, 30, replace=False), 'renda'] = np.nan
df_clientes.loc[np.random.choice(n, 15, replace=False), 'satisfacao'] = np.nan

print("=== PASSO 1: INSPECAO BASICA ===")
print(f"Shape: {df_clientes.shape}")
print(f"\nNulos:\n{df_clientes.isnull().sum()}")
print(f"\nDescribe:\n{df_clientes.describe().round(2)}")

print("\n=== PASSO 2: UNIVARIADA ===")
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, df_clientes.columns):
    df_clientes[col].hist(ax=ax, bins=20, alpha=0.7, edgecolor='black')
    ax.set_title(col)
plt.tight_layout()
plt.show()

print("\n=== PASSO 3: OUTLIERS EM RENDA ===")
Q1 = df_clientes['renda'].quantile(0.25)
Q3 = df_clientes['renda'].quantile(0.75)
IQR = Q3 - Q1
outliers = df_clientes[(df_clientes['renda'] < Q1-1.5*IQR) | (df_clientes['renda'] > Q3+1.5*IQR)]
print(f"Outliers em renda: {len(outliers)} ({len(outliers)/len(df_clientes)*100:.1f}%)")

print("\n=== PASSO 4: CORRELACAO COM CHURN ===")
corr_churn = df_clientes.corr()['churn'].abs().sort_values(ascending=False)
print("Features correlacionadas com churn:")
for feat, val in corr_churn[1:].items():
    print(f"  {feat}: {val:.3f}")

print("\n=== INSIGHTS ===")
print("1. Target churn e desbalanceado (~30% positivo)")
print("2. Renda tem distribuicao lognormal com outliers a direita")
print("3. Tempo como cliente e exponencial (muitos novos, poucos antigos)")
print("4. Nenhuma feature tem correlacao forte com churn (dados sinteticos)")

=== PASSO 1: INSPECAO BASICA ===
Shape: (500, 6)

Nulos:
idade                   0
renda                  30
tempo_cliente_meses     0
num_compras             0
satisfacao             15
churn                   0
dtype: int64

Describe:
        idade      renda  tempo_cliente_meses  num_compras  satisfacao   churn
count  500.00     470.00               500.00       500.00      485.00  500.00
mean    34.93   28614.65                24.47         4.98        6.84    0.29
std     11.12   22041.93                25.16         2.27        1.48    0.46
min     18.00    3334.84                 0.00         0.00        2.50    0.00
25%     26.00   14375.18                 7.00         3.00        5.90    0.00
50%     35.00   22456.19                17.00         5.00        6.90    0.00
75%     42.00   35225.93                32.00         6.00        7.80    1.00
max     80.00  139060.93               179.00        12.00       10.00    1.00

=== PASSO 2: UNIVARIADA ===



=== PASSO 3: OUTLIERS EM RENDA ===
Outliers em renda: 31 (6.2%)

=== PASSO 4: CORRELACAO COM CHURN ===
Features correlacionadas com churn:
  renda: 0.088
  idade: 0.077
  tempo_cliente_meses: 0.028
  satisfacao: 0.028
  num_compras: 0.013

=== INSIGHTS ===
1. Target churn e desbalanceado (~30% positivo)
2. Renda tem distribuicao lognormal com outliers a direita
3. Tempo como cliente e exponencial (muitos novos, poucos antigos)
4. Nenhuma feature tem correlacao forte com churn (dados sinteticos)


plt.show()


### Exercicio 2: Deteccao de Leakage

O dataset abaixo simula um cenario de churn com *data leakage intencional*.
Identifique quais features contem leakage e explique por que.

In [ ]:
# TAREFA DO ALUNO: Exercicio 2 - Detectar Leakage
np.random.seed(42)
n = 300
churn = np.random.binomial(1, 0.3, n)

df_leakage = pd.DataFrame({
    'idade': np.random.normal(35, 10, n).astype(int),
    'renda': np.random.lognormal(10, 0.5, n),
    'dias_desde_ultima_compra': np.where(churn == 1, np.random.exponential(90, n), np.random.exponential(15, n)),
    'motivo_cancelamento': np.where(churn == 1, np.random.choice(['preco', 'servico', 'concorrente'], n), 'ativo'),
    'score_satisfacao': np.clip(np.random.normal(7, 1, n) - churn * 3, 1, 10),
    'churn': churn
})

# TAREFA DO ALUNO: Calcular correlacao de cada feature com churn
# TAREFA DO ALUNO: Identificar features com leakage
# TAREFA DO ALUNO: Para cada feature suspeita, explicar se e leakage ou nao

leakage_features = None  # TAREFA DO ALUNO: lista de features com leakage
print(f"Features com leakage: {leakage_features}")

In [13]:
# SOLUCAO - Exercicio 2
np.random.seed(42)
n = 300
churn = np.random.binomial(1, 0.3, n)

df_leakage = pd.DataFrame({
    'idade': np.random.normal(35, 10, n).astype(int),
    'renda': np.random.lognormal(10, 0.5, n),
    'dias_desde_ultima_compra': np.where(churn == 1, np.random.exponential(90, n), np.random.exponential(15, n)),
    'motivo_cancelamento': np.where(churn == 1, np.random.choice(['preco', 'servico', 'concorrente'], n), 'ativo'),
    'score_satisfacao': np.clip(np.random.normal(7, 1, n) - churn * 3, 1, 10),
    'churn': churn
})

print("=== ANALISE DE LEAKAGE ===")
print("\nCorrelacoes com churn:")
for col in ['idade', 'renda', 'dias_desde_ultima_compra', 'score_satisfacao']:
    corr = df_leakage[col].corr(df_leakage['churn'])
    flag = ' *** SUSPEITO!' if abs(corr) > 0.3 else ''
    print(f"  {col}: {corr:.3f}{flag}")

print("\n=== VEREDICTO ===")
print("\n1. motivo_cancelamento: LEAKAGE DIRETO")
print("   -> So existe APOS o churn acontecer. Em producao, nao temos essa info.")
print("\n2. dias_desde_ultima_compra: LEAKAGE PROVAVEL")
print("   -> Se medido no momento do churn, contem info do futuro.")
print("   -> Se medido antes, pode ser feature legitima.")
print("\n3. score_satisfacao: LEAKAGE SUTIL")
print("   -> Se coletado ANTES do churn, e legitimo.")
print("   -> Se coletado DURANTE o processo de cancelamento, e leakage.")
print("   -> Correlacao alta (-0.6+) sugere leakage.")
print("\n4. idade, renda: OK (nao dependem do target)")

leakage_features = ['motivo_cancelamento', 'dias_desde_ultima_compra', 'score_satisfacao']
print(f"\nFeatures com leakage (ou suspeita): {leakage_features}")

=== ANALISE DE LEAKAGE ===

Correlacoes com churn:
  idade: 0.033
  renda: -0.049
  dias_desde_ultima_compra: 0.560 *** SUSPEITO!
  score_satisfacao: -0.825 *** SUSPEITO!

=== VEREDICTO ===

1. motivo_cancelamento: LEAKAGE DIRETO
   -> So existe APOS o churn acontecer. Em producao, nao temos essa info.

2. dias_desde_ultima_compra: LEAKAGE PROVAVEL
   -> Se medido no momento do churn, contem info do futuro.
   -> Se medido antes, pode ser feature legitima.

3. score_satisfacao: LEAKAGE SUTIL
   -> Se coletado ANTES do churn, e legitimo.
   -> Se coletado DURANTE o processo de cancelamento, e leakage.
   -> Correlacao alta (-0.6+) sugere leakage.

4. idade, renda: OK (nao dependem do target)

Features com leakage (ou suspeita): ['motivo_cancelamento', 'dias_desde_ultima_compra', 'score_satisfacao']


### Exercicio 3: Feature Engineering Orientado por EDA

A partir do dataset de clientes do Exercicio 1, crie pelo menos 3 novas features
baseadas nos insights da EDA. Calcule a correlacao de cada nova feature com o target.

In [ ]:
# TAREFA DO ALUNO: Exercicio 3 - Feature Engineering
np.random.seed(42)
n = 500
df_fe = pd.DataFrame({
    'idade': np.random.normal(35, 12, n).clip(18, 80).astype(int),
    'renda': np.random.lognormal(10, 0.7, n).round(2),
    'tempo_cliente_meses': np.random.exponential(24, n).round(0).astype(int),
    'num_compras': np.random.poisson(5, n),
    'satisfacao': np.clip(np.random.normal(7, 1.5, n), 1, 10).round(1),
    'churn': np.random.binomial(1, 0.3, n)
})

# TAREFA DO ALUNO: Criar pelo menos 3 novas features
# Sugestoes: renda_por_compra, cliente_novo, satisfacao_baixa, log_renda, compras_por_mes

# TAREFA DO ALUNO: Calcular correlacao das novas features com churn
print("Novas features e correlacao com churn:")

In [15]:
# SOLUCAO - Exercicio 3
np.random.seed(42)
n = 500
df_fe = pd.DataFrame({
    'idade': np.random.normal(35, 12, n).clip(18, 80).astype(int),
    'renda': np.random.lognormal(10, 0.7, n).round(2),
    'tempo_cliente_meses': np.random.exponential(24, n).round(0).astype(int),
    'num_compras': np.random.poisson(5, n),
    'satisfacao': np.clip(np.random.normal(7, 1.5, n), 1, 10).round(1),
    'churn': np.random.binomial(1, 0.3, n)
})

# Feature 1: renda por compra (poder de compra)
df_fe['renda_por_compra'] = df_fe['renda'] / (df_fe['num_compras'] + 1)

# Feature 2: cliente novo (menos de 6 meses)
df_fe['cliente_novo'] = (df_fe['tempo_cliente_meses'] < 6).astype(int)

# Feature 3: satisfacao baixa (abaixo de 5)
df_fe['satisfacao_baixa'] = (df_fe['satisfacao'] < 5).astype(int)

# Feature 4: log da renda (normalizar assimetria)
df_fe['log_renda'] = np.log1p(df_fe['renda'])

# Feature 5: compras por mes de relacionamento
df_fe['compras_por_mes'] = df_fe['num_compras'] / (df_fe['tempo_cliente_meses'] + 1)

# Correlacoes
print("=== NOVAS FEATURES E CORRELACAO COM CHURN ===")
novas = ['renda_por_compra', 'cliente_novo', 'satisfacao_baixa', 'log_renda', 'compras_por_mes']
originais = ['idade', 'renda', 'tempo_cliente_meses', 'num_compras', 'satisfacao']

print("\nFeatures originais:")
for col in originais:
    print(f"  {col}: {df_fe[col].corr(df_fe['churn']):.3f}")

print("\nNovas features:")
for col in novas:
    corr = df_fe[col].corr(df_fe['churn'])
    melhor = ' (melhor que original!)' if abs(corr) > 0.05 else ''
    print(f"  {col}: {corr:.3f}{melhor}")

print("\nInsight: em dados sinteticos, features engenheiradas tem correlacao")
print("similar as originais. Em dados reais, a melhoria costuma ser significativa.")

=== NOVAS FEATURES E CORRELACAO COM CHURN ===

Features originais:
  idade: -0.077
  renda: -0.072
  tempo_cliente_meses: 0.028
  num_compras: 0.013
  satisfacao: -0.033

Novas features:
  renda_por_compra: -0.062 (melhor que original!)
  cliente_novo: 0.012
  satisfacao_baixa: -0.018
  log_renda: -0.085 (melhor que original!)
  compras_por_mes: 0.023

Insight: em dados sinteticos, features engenheiradas tem correlacao
similar as originais. Em dados reais, a melhoria costuma ser significativa.


### O que observar nos exercicios

- O Exercicio 1 mostra que o framework de 7 passos funciona em *qualquer* dataset, nao apenas Titanic
- O Exercicio 2 revela que leakage pode ser obvio (motivo_cancelamento) ou sutil (score_satisfacao)
- O Exercicio 3 demonstra que feature engineering e um processo criativo guiado por EDA

### O que concluir dos exercicios

- **O framework EDA e universal**: mesmos 7 passos para e-commerce, saude, financas ou qualquer dominio
- **Leakage e o erro mais caro de ML**: gaste tempo extra verificando features suspeitas
- **Features engenheiradas precisam de validacao**: correlacao com target e necessaria mas nao suficiente (teste com CV)

### Conexao com outros notebooks

- O pipeline de feature engineering do Exercicio em questao usa as tecnicas deste proprio notebook
- A validacao de features com CV aparece em `3_0_tutorial_from_scratch`

### O que observar no panorama geral

- EDA e iterativa, nao linear: insights da bivariada podem levar voce a revisitar a univariada
- O relatorio EDA e o "contrato" entre exploracao e modelagem

### O que concluir do panorama geral

- **EDA nao e opcional**: pular EDA e a causa mais comum de modelos que falham em producao
- **Documente tudo**: "por que imputei age com mediana?" deve estar escrito em algum lugar

### Conexao com outros notebooks

- O framework EDA completo e aplicado em todos os projetos praticos de `6_1` a `6_4`
- A deteccao de leakage aqui previne problemas graves em `3_0_tutorial_from_scratch`

## 10. Erros Comuns e Armadilhas

### Erro 1: Confundir correlacao com causalidade
Idade correlaciona com sobrevivencia no Titanic, mas idade nao *causa* morte. O mecanismo
causal e: idade -> mobilidade -> capacidade de chegar aos botes. Sem DAG causal (ver
`1_5_design_experimentos`), nao assuma causalidade.

### Erro 2: Ignorar desbalanceamento do target
Se 95% dos emails sao nao-spam, um modelo que sempre diz "nao-spam" tem 95% de acuracia.
Verifique a distribuicao do target e use metricas adequadas (F1, AUC, precision-recall).

### Erro 3: Criar features usando informacao do target (leakage)
`mean_survival_by_class` parece uma boa feature... mas ela usa o target! Em producao,
voce nao sabe quem sobreviveu. Cuidado com features agregadas que incluem o target.

### Erro 4: Nao documentar decisoes de limpeza
"Por que removemos 77% de deck?" "Por que mediana e nao media para age?" Sem documentacao,
ninguem (incluindo voce no futuro) consegue reproduzir ou auditar o processo.

### Erro 5: EDA no dataset completo antes do split
Se voce faz EDA nos dados de teste, suas decisoes de feature engineering estao "contaminadas"
por informacao do teste. Idealmente, faca EDA so no treino e aplique as mesmas transformacoes
no teste.

### Erro 6: Visualizacoes sem contexto
Um grafico de barras sem titulo, sem labels nos eixos e sem legenda e inutil. Sempre
adicione contexto: "Taxa de sobrevivencia por sexo (Titanic, N=891)".

### Erro 7: Ignorar outliers e assumir que o modelo "se vira"
Modelos lineares sao sensíveis a outliers; arvores de decisao nao. A decisao de tratar
outliers depende do modelo que voce planeja usar.

## 11. Resumo e Conexoes

### Hierarquia de Conceitos

```
DATASET BRUTO
    |
    v
1. PERGUNTAS (o que quero prever? quais features?)
    |
    v
2. INSPECAO (shape, dtypes, nulos, describe, head)
    |
    v
3. UNIVARIADA (histogramas, barras, estatisticas por feature)
    |
    v
4. BIVARIADA (correlacao, boxplots por grupo, crosstabs)
    |
    v
5. MULTIVARIADA (pair plots, heatmaps, interacoes)
    |
    v
6. PROBLEMAS
    |---> Outliers: IQR, z-score, dominio
    |---> Nulos: MCAR/MAR/MNAR, estrategia por coluna
    |---> Leakage: feature derivada do target? info do futuro?
    |
    v
7. INSIGHTS E RECOMENDACOES
    |---> Features para manter/remover
    |---> Transformacoes necessarias
    |---> Feature engineering sugerido
    |---> Metrica recomendada
    |
    v
RELATORIO EDA -> MODELAGEM
```

### Tabela de Conexoes

| Conceito EDA | Notebook anterior | Notebook futuro |
|--------------|------------------|-----------------|
| Estatisticas descritivas | `1_1_estatistica_descritiva` | EDA visual e numerica (este notebook) |
| Correlacao | `1_2_estatistica_inferencial` | `1_4_regressao_estatistica` (VIF) |
| Data leakage | `1_5_design_experimentos` (causalidade) | `3_0_tutorial_from_scratch` |
| Feature engineering | `2_1_python_data_science` (Pandas) | Criacao de features (este notebook) |
| Visualizacao | `2_1_python_data_science` (Matplotlib) | `6_1_deploy_modelos` |
| Outliers | `1_1_estatistica_descritiva` (IQR) | Detecao multivariada (este notebook) |

### Checklist de Competencias

- [ ] Sei aplicar o framework de 7 passos de EDA em qualquer dataset
- [ ] Sei criar visualizacoes univariadas e bivariadas informativas
- [ ] Sei detectar e tratar outliers com metodo IQR
- [ ] Sei analisar padroes de valores ausentes
- [ ] Sei identificar data leakage em features
- [ ] Sei criar features exploratórias e testar correlacao com target
- [ ] Sei gerar um relatorio EDA com recomendacoes acionaveis

### Proximos Passos

1. **`2_3_sql_e_apis`**: Acessar dados de fontes reais (SQL, APIs)
2. Tecnicas avancadas de criacao de features (proximas secoes deste notebook)
3. **`4_1_fundamentos_redes_neurais`**: Implementar as recomendacoes do relatorio EDA no pipeline (`3_0_tutorial_from_scratch`)